# RynnVLA-002 — five ablations, decoder-first

The mouth is settled (2026-09-04: prompt echo → 2-token loop, `garbage`). No more prompt-hacking beyond one fair rung — that was the Molmo lesson. These rungs are about the part you keep: the VQ decoder and the world model.

| Rung | Question | Kill criterion |
|---|---|---|
| 1 | **Recon floor.** Tokenize→decode the current frames, no LM. | If recon is mush, dream quality can never beat it — judge rungs 2–4 against this, not the raw PNG. |
| 2 | **Action counterfactuals.** Same frame, three very different actions. | Near-identical next frames ⇒ the action is ignored ⇒ "world model" is a video prior and the overnight/dream score is hollow. |
| 3 | **Open-loop rollout ×3.** Feed the dreamed frames back in. | Collapse by step 3 ⇒ no usable dream data loop. |
| 4 | **Unified card.** Does `VLA_model_256` (the acting card) also dream? Its `abiw` training mix includes 256 world data. | If yes: one 14 GB card acts *and* dreams. If no: two-card story stands. |
| 5 | **On-railroad mouth.** Official conversation format (sep `8710`) through ItemProcessor, free decode + banned-BPE decode. | Loop / action tokens again ⇒ mouth verdict final at every distance from the SFT distribution. |

Kernel **rynn-worldvla**. One 7B resident at a time; cells manage the unloads. Rungs 2–3 are ~2 min per dream at 512² (six dreams total); budget ~20 min for the ladder.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import torch
from IPython.display import display, Markdown
from transformers import GenerationConfig

from rynn_turns import (
    RynnSession, load_sample_cameras, append_log, leftover_verdict,
    _on_path, _ensure_hf_tokenizer_dir, _rynnvla_cwd, _extract_image_blocks,
    decode_generated_images, _target_size_of,
    SEP_ID, PAD, IMG_LO, IMG_HI, LOG_DIR,
)

third, wrist, third_p, wrist_p = load_sample_cameras()
sess = RynnSession()
_on_path()
from data.pre_tokenize_action import ItemProcessor as ItemProcessorAction  # no <|state|>

with _rynnvla_cwd():
    proc512 = ItemProcessorAction(tokenizer=str(_ensure_hf_tokenizer_dir()), target_size=512)
    proc256 = ItemProcessorAction(tokenizer=str(_ensure_hf_tokenizer_dir()), target_size=256)

WORLD_PROMPT = (
    "Generate the next image based on the provided sequence of "
    "historical images and corresponding actions."
)

def gen_ids(model, tokens, max_new=3000):
    """Greedy generate on the official class; KV freed before return."""
    from model.chameleon import ChameleonForConditionalGeneration
    model.init_input_ids = None
    cfg = GenerationConfig(
        max_new_tokens=max_new, max_length=model.config.max_position_embeddings,
        temperature=1, top_k=None, do_sample=False,
        eos_token_id=[SEP_ID], pad_token_id=PAD,
    )
    ids = torch.tensor(tokens, dtype=torch.int64, device=model.device).unsqueeze(0)
    with torch.inference_mode():
        res = ChameleonForConditionalGeneration.generate(
            model, input_ids=ids, attention_mask=torch.ones_like(ids),
            generation_config=cfg, return_dict_in_generate=True,
        )
    out = res["sequences"][0, ids.shape[1]:].detach().cpu().tolist()
    del res
    model.init_input_ids = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

def decode_blocks(proc, ids, tag):
    """Official decode_image assumes a perfect 1060-token 512² span.

    Rung 2 emits extra short 8197…8196 glitches and near-miss frames
    (1028 = no newlines, 1059 = one short → KeyError 8803). Recover those;
    skip junk spans; if 8196 closed too early, stitch leftover VQ tokens.
    """
    target = _target_size_of(proc)
    imgs, stats = decode_generated_images(proc, ids, target_size=target, n_want=2)
    extra = f" notes={stats['notes']}" if stats.get("notes") else ""
    print(
        f"[{tag}] new={stats['n_new']} img_range={stats['n_img_range']} "
        f"blocks={stats['block_lens']} chosen={stats['chosen_lens']} "
        f"errs={stats['errors']}{extra}"
    )
    return imgs, {"tag": tag, **stats}

def dream_pair(model, proc, cur_third, cur_wrist, action, tag, max_new=3000):
    """002 trained world prompt: <|image|><|image|><|action|> → next third+wrist."""
    conv = {
        "conversations": [{"from": "human", "value": WORLD_PROMPT + "<|image|><|image|><|action|>"}],
        "image": [cur_third, cur_wrist],
        "action": [np.asarray(action, dtype=np.float64)],
    }
    tokens = proc.process_item(conv, training_mode=False)
    ids = gen_ids(model, tokens, max_new)
    imgs, stats = decode_blocks(proc, ids, tag)
    stats["n_prompt"] = len(tokens)
    append_log({"sandbox": "rynn_worldvla", "turn": "ablation", **stats,
                "action": [float(x) for x in action]})
    return imgs, stats

def show(img, title):
    display(Markdown(f"**{title}**"))
    display(img)

print("ready — third", third.size, "wrist", wrist.size)

## Rung 1 — VQ recon floor (no LM)

Tokenize→decode the *current* frames through Meta VQGAN at both resolutions. This is the ceiling for every dream: blur or artifacts here are the codec, not the world model. No 7B is loaded for this rung.

In [ ]:
def recon(proc, img, tag):
    conv = {"conversations": [{"from": "human", "value": "<|image|>"}],
            "image": [img], "action": []}
    tokens = proc.process_item(conv, training_mode=False)
    block = _extract_image_blocks(tokens)[0]
    out = proc.decode_image(list(block))
    show(out, f"recon {tag} ({len(block)} tokens)")
    return out

display(Markdown("**Originals**"))
show(third, "third (input)")
show(wrist, "wrist (input)")
recon(proc512, third, "third @512")
recon(proc512, wrist, "wrist @512")
recon(proc256, third, "third @256")
recon(proc256, wrist, "wrist @256")
append_log({"sandbox": "rynn_worldvla", "turn": "ablation", "tag": "rung1/recon",
            "note": "VQ floor saved visually in notebook; judge rungs 2-4 against this"})

## Rung 2 — action counterfactuals (world card, 512)

Same frame pair, three env-space actions that should look very different one step later: push left, push right, gripper toggle. Raw env actions; `process_action` normalizes internally.

Read the three third-view frames side by side. **Kill:** if they are near-identical, the action tokens are decoration and the world model is a video prior.

In [ ]:
model_w = sess._ensure_world()

ACTS = {
    "push_left":  [-0.8, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
    "push_right": [ 0.8, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
    "grip_close": [ 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,  1.0],
}

rung2 = {}
for tag, a in ACTS.items():
    imgs, stats = dream_pair(model_w, proc512, third, wrist, a, f"rung2/{tag}")
    rung2[tag] = imgs
    for j, im in enumerate(imgs):
        show(im, f"{tag} → {'third' if j == 0 else 'wrist'}")

## Rung 3 — open-loop rollout ×3 (world card, 512)

Feed the dreamed third+wrist back in with the same action. This is the overnight-loop question: WorldVLA's paper claim is exactly this autoregressive rollout.

**Kill:** collapse (mush, wrong scene, malformed blocks) by step 3 ⇒ no self-generated data loop; the `overnight` score rests on their curated pipeline only.

In [ ]:
cur_t, cur_w = third, wrist
a = ACTS["push_left"]
for step in range(3):
    imgs, stats = dream_pair(model_w, proc512, cur_t, cur_w, a, f"rung3/step{step}")
    if len(imgs) < 2:
        print(f"rollout broke at step {step}: {stats['errors']}")
        break
    cur_t, cur_w = imgs[0], imgs[1]
    show(cur_t, f"rollout step {step} — third")
    show(cur_w, f"rollout step {step} — wrist")

## Rung 4 — does the acting card dream? (VLA card, 256)

`VLA_model_256/libero_goal` was trained on the `abiw` mix — action data **and** 256 world data (`concate_action_world_model_data_libero.py`: `…w_state_5_256` + `…a2i_256`). If the same 14 GB card that emitted the five action chunks also produces a coherent next frame, the unified-model claim holds on the exact checkpoint you'd deploy, and the second card is optional.

World card is unloaded first; ~600-token prompt, ~560-token generation — under the VLA's 4096 ctx.

In [ ]:
sess.unload_world()
model_v = sess._ensure_vla()

imgs, stats = dream_pair(model_v, proc256, third, wrist, ACTS["push_left"], "rung4/vla256", max_new=1200)
if imgs:
    for j, im in enumerate(imgs):
        show(im, f"VLA-card dream → {'third' if j == 0 else 'wrist'}")
else:
    print("VLA card did not produce decodable image blocks:", stats)

## Rung 5 — on-railroad mouth (VLA card, one fair shot)

The 09-04 talk probe was a *bare string* through `LlamaTokenizerFast` — not the trained turn format. The fair version: a text+image question through the official `ItemProcessor` conversation (turn ends with sep `8710`, image as VQ tokens), which is exactly how every training sample was shaped.

Two decodes: **free** (what it *wants* to emit — action bins? image tokens? English?) and **banned** (only BPE ≥16384 allowed — best case).

**Kill:** loop or non-BPE flood here too ⇒ the mouth verdict is final; log it and stop. No rung 6.

In [ ]:
question = "What objects are on the table? Answer in English."
conv = {"conversations": [{"from": "human", "value": question + "<|image|>"}],
        "image": [third], "action": []}
tokens = proc256.process_item(conv, training_mode=False)

free_ids = gen_ids(model_v, tokens, max_new=64)
free_txt = sess.tokenizer.decode(free_ids, skip_special_tokens=False)
n_bpe = sum(1 for i in free_ids if i >= 16384)
n_img = sum(1 for i in free_ids if IMG_LO <= i <= IMG_HI)
n_act = sum(1 for i in free_ids if 10004 <= i <= 15004)
print(f"free decode: {len(free_ids)} ids — bpe {n_bpe}, img {n_img}, action-bin {n_act}")
print("free ids:", free_ids[:32])
print("free text:", repr(free_txt[:300]))

ban_ids, ban_txt = sess._greedy_banned(model_v, tokens)
verdict = leftover_verdict(ban_txt, ban_ids)
print("banned ids:", ban_ids[:32])
print("banned text:", repr(ban_txt[:300]))
print("verdict:", verdict)

append_log({"sandbox": "rynn_worldvla", "turn": "ablation", "tag": "rung5/onrail_mouth",
            "question": question,
            "free": {"ids": free_ids[:48], "text": free_txt[:300],
                     "n_bpe": n_bpe, "n_img": n_img, "n_act": n_act},
            "banned": {"ids": ban_ids[:48], "text": ban_txt[:300]},
            "verdict": verdict})

## Verdicts (fill after the run, then log)

| Rung | Result | Meaning |
|---|---|---|
| 1 recon floor | | |
| 2 counterfactuals | | action-conditioned or video prior |
| 3 rollout ×3 | | dream data loop viable? |
| 4 unified card | | one card or two |
| 5 on-rail mouth | | final mouth verdict |

Write the outcome into `notes/10_sibling_list.md` (pull log, `rynn-worldvla (2026-09-04)`) and sandbox C in `notes/09_next_session.md`. Everything is also in `logs/*.jsonl` under `"turn": "ablation"`.